In [ ]:
%run common_deployment_config

In [ ]:
# Notebook-specific imports for environment deployment
import time
import io
import zipfile
import hashlib

print("✓ Notebook-specific imports loaded")


---

## Configuration Parameters

**Configure your deployment settings below:**

Modify the following variables to match your environment:

### Lakehouse Connection Components:
- `WORKSPACE_NAME`: OneLake workspace name (configured in common_deployment_config)
- `ENDPOINT`: OneLake endpoint (auto-detected from Fabric runtime)
- `ARTIFACT_LAKEHOUSE_NAME`: Lakehouse containing artifacts (configured in common_deployment_config)

### Deployment Configuration:
- `ENVIRONMENT_NAME`: Name of the Fabric Environment (auto-prefixed with COMPANY_PREFIX and technical_prefix)
- `LIBRARY_CAPABILITY`: Capability/library folder name (e.g., "healthcare-libraries")
- `LIBRARY_VERSION`: Version from ARTIFACT_VERSION in common config
- `PACKAGE_PATTERNS`: List of patterns to match WHL files (e.g., ["hds", "dtt"])

> **Note:** The `LAKEHOUSE_BASE_PATH` will be automatically constructed from the lakehouse components.

> **⚠️ Important:** The deployment will **fail** if an environment with the specified name already exists. Please use a unique environment name or delete the existing environment first.


In [ ]:
# ============================================================================
# NOTEBOOK-SPECIFIC CONFIGURATION
# ============================================================================
# (Common config loaded from common_deployment_config)

# Environment-specific settings
# Apply prefixes to environment name: {COMPANY_PREFIX}_{TECHNICAL_PREFIX}_{TARGET_ENVIRONMENT_NAME}
ENVIRONMENT_NAME = build_artifact_name(TARGET_ENVIRONMENT_NAME)
# ENVIRONMENT_NAME = "My Custom Environment"  # Uncomment to override with exact name (no prefixes)

print("✓ Environment deployer configuration:")
print(f"  Workspace: {WORKSPACE_NAME}")
print(f"  Endpoint: {ENDPOINT_URI}")
print(f"  Base Environment Name: {TARGET_ENVIRONMENT_NAME}")
print(f"  Prefixed Environment: {ENVIRONMENT_NAME}")
print(f"  Version: {ARTIFACT_VERSION}")

---

## Step 1: Configuration & Setup

Configure the deployment parameters and discover available libraries using the configuration function.

**Configuration Function:** `setup_deployment_configuration()`

**Parameters:**
- `environment_name`: Name of the Fabric Environment to create/update
- `lakehouse_base_path`: Base ABFS path to the lakehouse (up to /dist folder)
- `library_capability`: Capability/library folder name (e.g., "healthcare-libraries")
- `library_version`: Version number of the libraries
- `package_patterns`: List of patterns to match WHL files (e.g., ["hds", "dtt"])
- `workspace_id`: Target workspace ID (auto-detected if not provided)

**Returns:**
- Configuration dictionary with all deployment parameters and discovered WHL packages

**Validation:**
- ✓ Validates library artifacts path exists
- ✓ Scans for WHL packages matching specified patterns
- ✗ **Raises exception if no WHL packages are found** (deployment stops here)
- ✗ **Raises exception if library artifacts path doesn't exist** (deployment stops here)


In [ ]:
def setup_deployment_configuration(
    environment_name: str,
    lakehouse_base_path: str,
    library_capability: str,
    library_version: str,
    package_patterns: list = None,
    workspace_id: Optional[str] = None
) -> Dict[str, Any]:
    """
    Configure deployment parameters and discover available WHL libraries.

    Args:
        environment_name: Name of the Fabric Environment to create/update
        lakehouse_base_path: Base ABFS path to the lakehouse (up to /dist folder)
        library_capability: Capability/library folder name (e.g., "healthcare-libraries")
        library_version: Version number of the libraries
        package_patterns: List of patterns to match WHL files (default: ["hds", "dtt"])
        workspace_id: Target workspace ID (auto-detected if None)

    Returns:
        Dictionary containing configuration parameters and discovered packages

    Raises:
        FileNotFoundError: If library artifacts path does not exist
        ValueError: If no WHL packages are found matching the patterns
    """
    # Use workspace ID from common config (already auto-detected)
    if workspace_id is None or workspace_id =='':
        workspace_id = WORKSPACE_ID

    # Set default package patterns if not provided
    if package_patterns is None or len(package_patterns) == 0:
        package_patterns = ["hds", "dtt"]

    # Construct full library artifacts path
    library_artifacts_path = f"{lakehouse_base_path}/{library_capability}/{library_version}"

    print(f"✓ Detected Workspace ID: {workspace_id}")
    print(f"\n✓ Configuration:")
    print(f"  Workspace ID: {workspace_id}")
    print(f"  Environment Name: {environment_name}")
    print(f"  Lakehouse Base Path: {lakehouse_base_path}")
    print(f"  Library Capability: {library_capability}")
    print(f"  Library Version: {library_version}")
    print(f"  Library Artifacts Path: {library_artifacts_path}")

    # Validate library artifacts path exists
    if not notebookutils.fs.exists(library_artifacts_path):
        error_msg = f"Library artifacts path does not exist: {library_artifacts_path}"
        print(f"\n✗ ERROR: {error_msg}")
        raise FileNotFoundError(error_msg)

    print(f"  ✓ Library artifacts path exists")

    # Discover WHL packages
    discovered_whl_packages = []
    try:
        all_files = [f.name for f in notebookutils.fs.ls(library_artifacts_path) if not f.isDir]
        for file in all_files:
            if file.endswith(".whl"):
                file_lower = file.lower()
                for pattern in package_patterns:
                    if pattern in file_lower:
                        discovered_whl_packages.append(file)
                        break
        discovered_whl_packages.sort()

        print(f"\n✓ Discovered {len(discovered_whl_packages)} WHL package(s):")
        for pkg in discovered_whl_packages:
            print(f"    - {pkg}")

        # Raise exception if no packages found
        if len(discovered_whl_packages) == 0:
            error_msg = f"No WHL packages found matching patterns: {package_patterns}"
            print(f"\n✗ ERROR: {error_msg}")
            raise ValueError(error_msg)

    except (FileNotFoundError, ValueError):
        # Re-raise validation errors
        raise
    except Exception as e:
        error_msg = f"Error discovering WHL packages: {str(e)}"
        print(f"\n✗ ERROR: {error_msg}")
        raise RuntimeError(error_msg) from e

    # Return configuration dictionary
    return {
        "workspace_id": workspace_id,
        "environment_name": environment_name,
        "lakehouse_base_path": lakehouse_base_path,
        "library_capability": library_capability,
        "library_version": library_version,
        "library_artifacts_path": library_artifacts_path,
        "discovered_whl_packages": discovered_whl_packages,
        "package_patterns": package_patterns
    }

In [ ]:
# Call the configuration function with deployment parameters
config = setup_deployment_configuration(
    environment_name=ENVIRONMENT_NAME,
    lakehouse_base_path=BASE_DIST_PATH,
    library_capability="healthcare-libraries",  # From path structure
    library_version=ARTIFACT_VERSION,
    package_patterns=LIBRARY_PACKAGE_PATTERNS
)

# Extract configuration values for easy access
workspace_id = config["workspace_id"]
environment_name = config["environment_name"]
library_artifacts_path = config["library_artifacts_path"]
discovered_whl_packages = config["discovered_whl_packages"]

print("\n" + "=" * 60)
print("✓ Configuration completed successfully!")
print("=" * 60)


---

## Step 2: Function Definitions

This section defines all helper functions used in the deployment workflow.

### 2.1 Environment Management Functions
- `get_environment_by_name()`: Retrieve environment ID by name
- `create_environment()`: Create a new Fabric Environment
- `get_or_create_environment()`: Check if environment exists (raises exception if found), otherwise create new environment
- `publish_environment()`: Publish environment to apply staged changes

### 2.2 Library Upload Functions
- `upload_pip_dependencies_from_yaml()`: Upload pip dependencies from environment.yml
- `upload_custom_whl_library_to_environment()`: Upload custom WHL/JAR/PY/TAR.GZ library files with validation

### 2.3 Validation & Utility Functions
- `_read_file_as_binary()`: Read file as binary using Spark
- `_calculate_sha256_hash()`: Calculate SHA256 hash for integrity validation
- `_validate_wheel_zip_structure()`: Verify WHL file is valid ZIP archive
- `_extract_library_names()`: Parse library names from staging metadata
- `_get_all_staging_library_names()`: Get all library names from staging with pagination support


In [ ]:
def get_environment_by_name(workspace_id: str, environment_name: str) -> Optional[str]:
    """
    Get environment ID by name from a workspace.
    """
    fabric_client = FabricRestClient()
    url = f"/v1/workspaces/{workspace_id}/environments"
    try:
        resp = fabric_client.get(url)
        if resp.ok:
            environments = resp.json().get("value", [])
            for env in environments:
                if env.get("displayName") == environment_name:
                    return env.get("id")
            return None
        else:
            print(f"  ✗ Failed to list environments")
            print(f"    Status: {resp.status_code}")
            return None
    except Exception as e:
        print(f"  ✗ Error listing environments: {str(e)}")
        return None

def create_environment(workspace_id: str, environment_name: str, description: Optional[str] = None) -> Optional[str]:
    """
    Create a new Fabric Environment in the workspace.
    """
    fabric_client = FabricRestClient()
    url = f"/v1/workspaces/{workspace_id}/environments"
    payload: Dict[str, Any] = {"displayName": environment_name}
    if description:
        payload["description"] = description
    try:
        resp = fabric_client.post(url, json=payload)
        if resp.ok:
            body = resp.json()
            env_id = body.get("id")
            if env_id:
                print(f"  ✓ Created new environment: {environment_name}")
                return env_id
            else:
                print(f"  ✗ Environment created but ID not found in response")
                return None
        else:
            print(f"  ✗ Failed to create environment")
            print(f"    Status: {resp.status_code}")
            try:
                error_detail = resp.json()
                print(f"    Error: {error_detail}")
            except:
                print(f"    Response: {resp.content}")
            return None
    except Exception as e:
        print(f"  ✗ Error creating environment: {str(e)}")
        return None

def get_or_create_environment(workspace_id: str, environment_name: str, description: Optional[str] = None) -> Optional[str]:
    """
    Check if environment exists and raise exception if it does, otherwise create it.

    Raises:
        ValueError: If an environment with the given name already exists
    """
    print(f"\nChecking for environment: {environment_name}")
    env_id = get_environment_by_name(workspace_id, environment_name)
    if env_id:
        error_msg = f"Environment '{environment_name}' already exists with ID: {env_id}. Please use a different name or delete the existing environment."
        print(f"\033[1;31m✗ ERROR: {error_msg}\033[0m")
        # raise ValueError(error_msg)
    else:
        print(f"  ℹ Environment does not exist, creating...")
        env_id = create_environment(workspace_id, environment_name, description)
        if env_id:
            print(f"    Environment ID: {env_id}")
        return env_id

def upload_pip_dependencies_from_yaml(workspace_id: str, environment_id: str, yaml_file_path: str) -> bool:
    """
    Upload pip dependencies to the Fabric Environment from an environment.yml file.
    """
    fabric_client = FabricRestClient()
    url = f"/v1/workspaces/{workspace_id}/environments/{environment_id}/staging/libraries/importExternalLibraries"
    try:
        env_yaml = notebookutils.fs.head(yaml_file_path, 1024 * 512)  # up to 512KB
        resp = fabric_client.request(
            "POST", url,
            data=env_yaml,
            headers={"Content-Type": "application/octet-stream"}
        )
        if resp.status_code < 400:
            print(f"  ✓ Pip packages staged from environment.yml")
            return True
        else:
            print(f"  ✗ Pip package upload failed")
            print(f"    Status: {resp.status_code}")
            print(f"    Response: {resp.text}")
            return False
    except Exception as e:
        print(f"  ✗ Error uploading pip packages: {str(e)}")
        return False

def publish_environment(workspace_id: str, environment_id: str, environment_name: str) -> bool:
    """
    Publish the Fabric Environment to apply all staged changes.
    """
    fabric_client = FabricRestClient()
    url = f"/v1/workspaces/{workspace_id}/environments/{environment_id}/staging/publish"
    publish_response = fabric_client.post(url)
    if publish_response.ok:
        print(f"\n✓ Environment '{environment_name}' publish initiated")
        print("  This operation can take several minutes to complete.")
        print("  Please check the Fabric portal to confirm publishing is complete before using this environment.")
        return True
    else:
        print(f"\n✗ Publish failed for environment '{environment_name}'")
        print(f"  Status: {publish_response.status_code}")
        try:
            error_detail = publish_response.json()
            print(f"  Error: {error_detail}")
        except:
            print(f"  Response: {publish_response.content}")
        return False


In [ ]:
def _read_file_as_binary(path: str) -> bytes:
    """Reads a file as raw bytes using Spark's binaryFile data source."""
    df = spark.read.format("binaryFile").load(path).select("content", "length")
    row = df.first()
    if row is None:
        raise FileNotFoundError(f"Could not read file as binary: {path}")
    return bytes(row["content"])

def _calculate_sha256_hash(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def _validate_wheel_zip_structure(wheel_bytes: bytes) -> None:
    """Ensures the WHL is a valid ZIP archive (a wheel is a ZIP)."""
    with zipfile.ZipFile(io.BytesIO(wheel_bytes), "r") as zf:
        bad_file = zf.testzip()
        if bad_file is not None:
            raise ValueError(f"Wheel ZIP is corrupted. First bad entry: {bad_file}")

def _extract_library_names(meta: dict) -> set:
    libs = meta.get("libraries")
    if libs is None:
        libs = meta.get("value", [])
    names = set()
    for lib in libs or []:
        n = lib.get("name") or lib.get("fileName")
        if n:
            names.add(n)
    return names

def _get_all_staging_library_names(fabric_client, workspace_id: str, environment_id: str, beta: bool) -> set:
    base_url = f"/v1/workspaces/{workspace_id}/environments/{environment_id}/staging/libraries?beta={str(beta).lower()}"
    all_names = set()
    continuation = None
    while True:
        url = base_url if not continuation else f"{base_url}&continuationToken={continuation}"
        resp = fabric_client.get(url)
        if not getattr(resp, "ok", False):
            return all_names
        meta = resp.json()
        all_names |= _extract_library_names(meta)
        continuation = meta.get("continuationToken")
        if not continuation or str(continuation).lower() == "null":
            break
    return all_names

def upload_custom_whl_library_to_environment(
    workspace_id: str,
    environment_id: str,
    library_file_name: str,
    library_file_path: str,
    beta: bool = False,
    verify_retries: int = 10,
    verify_sleep_seconds: int = 6,
    validate_wheel_zip: bool = True
) -> bool:
    """
    Upload custom WHL library (.whl/.jar/.py/.tar.gz) to Fabric Environment staging.

    Returns True if upload succeeds and library appears in staging list.
    """
    fabric_client = FabricRestClient()
    upload_url = (
        f"/v1/workspaces/{workspace_id}"
        f"/environments/{environment_id}"
        f"/staging/libraries/{library_file_name}"
        f"?beta={str(beta).lower()}"
    )
    print(f"url: {upload_url}")

    if not notebookutils.fs.exists(library_file_path):
        print(f"  ✗ File not found: {library_file_path}")
        return False

    try:
        content = _read_file_as_binary(library_file_path)
        file_size = len(content)
        file_hash = _calculate_sha256_hash(content)
        print(f"    Read {file_size} bytes from {library_file_path}")
        print(f"    SHA256: {file_hash}")
        if validate_wheel_zip and library_file_name.endswith(".whl"):
            _validate_wheel_zip_structure(content)
            print("    ✓ Wheel sanity check: valid ZIP structure")
        resp = fabric_client.post(upload_url, data=content, headers={"Content-Type": "application/octet-stream"})
        if resp.status_code >= 400:
            print(f"  ✗ Upload failed: {library_file_name} (HTTP {resp.status_code})")
            try:
                print(f"    Error: {resp.json()}")
            except Exception:
                print(f"    Response: {resp.content}")
            return False
        print(f"  ✓ Upload POST response: Success for {library_file_name}")
        for attempt in range(1, verify_retries + 1):
            print(f"    Verifying in libraries list... (attempt {attempt})")
            names = _get_all_staging_library_names(fabric_client, workspace_id, environment_id, beta)
            if library_file_name in names:
                print(f"    ✓ Verified: {library_file_name} present in staging libraries.")
                print(f"    ✓ Upload Validation Passed (size={file_size}, sha256={file_hash})")
                return True
            time.sleep(verify_sleep_seconds)
        print("    ✗ ERROR: Uploaded file not found in libraries list after retries.")
        return False
    except Exception as e:
        print(f"  ✗ Error uploading {library_file_name}: {str(e)}")
        return False


---

## Step 3: Main Orchestration Functions

This section defines the step-by-step deployment workflow functions and the main orchestration function.

### 3.1 Step Functions (Each deployment phase is a dedicated function)

**Step 1: Environment Setup**
- `step1_setup_environment()`: Check if environment exists (raises exception if found), otherwise create new environment
- Returns: environment_id and step results
- **Raises exception if environment already exists** (deployment stops here)

**Step 2: WHL Package Upload**
- `step2_upload_whl_packages()`: Upload custom WHL packages with validation
- Returns: upload results for each WHL package

**Step 3: Pip Package Upload**
- `step3_upload_pip_packages()`: Upload dependencies from environment.yml
- Returns: pip upload success status

**Step 4: Environment Publishing**
- `step4_publish_environment()`: Publish all staged changes to activate libraries
- Returns: publish success status

### 3.2 Main Orchestration Function

**`deploy_libraries_to_environment()`**
- Orchestrates the complete deployment by calling all step functions in sequence
- Aggregates results from all steps
- Provides comprehensive deployment summary

**Parameters:**
- `workspace_id`: Target workspace ID
- `environment_name`: Environment name
- `library_artifacts_path`: Library artifacts directory path containing WHL packages and environment.yml
- `discovered_whl_packages`: List of WHL package filenames to upload
- `environment_description`: Optional description

**Returns:**
- Dictionary with deployment status, success flags, and detailed results


In [ ]:
def step1_setup_environment(workspace_id: str, environment_name: str, environment_description: Optional[str] = None) -> Dict[str, Any]:
    """
    Step 1: Environment Setup
    Check if environment exists and create it if it doesn't.

    Args:
        workspace_id: Target workspace ID
        environment_name: Name of the environment
        environment_description: Optional environment description

    Returns:
        Dictionary with environment_id and step success status

    Raises:
        ValueError: If an environment with the given name already exists
    """
    print("=" * 60)
    print("STEP 1: Environment Setup")
    print("=" * 60)

    environment_id = get_or_create_environment(
        workspace_id=workspace_id,
        environment_name=environment_name,
        description=environment_description
    )

    if not environment_id:
        print("\n✗ CRITICAL ERROR: Could not get or create environment")
        print("  Cannot proceed with library uploads.")
        return {"environment_id": None, "success": False}

    print(f"\n✓ Environment ready for library uploads")
    print("-" * 60)

    return {"environment_id": environment_id, "success": True}


def step2_upload_whl_packages(workspace_id: str, environment_id: str, library_artifacts_path: str, discovered_whl_packages: list) -> Dict[str, Any]:
    """
    Step 2: Upload WHL Packages
    Upload custom WHL packages with validation.

    Args:
        workspace_id: Target workspace ID
        environment_id: Environment ID
        library_artifacts_path: Path to library artifacts directory
        discovered_whl_packages: List of WHL package filenames

    Returns:
        Dictionary with upload results for each package
    """
    print("\n" + "=" * 60)
    print("STEP 2: Uploading WHL Packages")
    print("=" * 60)

    whl_uploads = []

    if len(discovered_whl_packages) == 0:
        print("\n⚠ No WHL packages found to upload. Skipping WHL uploads.")
    else:
        for whl_file in discovered_whl_packages:
            whl_path = f"{library_artifacts_path}/{whl_file}"
            print(f"\nUploading: {whl_file}")
            success = upload_custom_whl_library_to_environment(
                workspace_id=workspace_id,
                environment_id=environment_id,
                library_file_name=whl_file,
                library_file_path=whl_path
            )
            whl_uploads.append({"filename": whl_file, "success": success})

        print("\n" + "-" * 60)
        print("WHL Upload Summary:")
        for upload in whl_uploads:
            status = "✓ Success" if upload["success"] else "✗ Failed"
            print(f"  {status}: {upload['filename']}")
        print("-" * 60)

    return {"whl_uploads": whl_uploads, "success": True}

def step3_upload_pip_packages(workspace_id: str, environment_id: str, library_artifacts_path: str) -> Dict[str, Any]:
    """
    Step 3: Upload Pip Packages
    Upload pip dependencies from environment.yml.

    Args:
        workspace_id: Target workspace ID
        environment_id: Environment ID
        library_artifacts_path: Path to library artifacts directory

    Returns:
        Dictionary with pip upload success status
    """
    print("\n" + "=" * 60)
    print("STEP 3: Uploading Pip Packages from environment.yml")
    print("=" * 60)

    env_yaml_path = f"{library_artifacts_path}/environment.yml"
    print(f"\nYAML file: {env_yaml_path}")

    if not notebookutils.fs.exists(env_yaml_path):
        print(f"\n⚠ WARNING: environment.yml not found at {env_yaml_path}")
        print("  Skipping pip package upload")
        return {"pip_upload": False, "success": True}

    pip_success = upload_pip_dependencies_from_yaml(
        workspace_id=workspace_id,
        environment_id=environment_id,
        yaml_file_path=env_yaml_path
    )

    if pip_success:
        print("\n✓ All pip packages uploaded successfully")
    else:
        print("\n✗ Pip package upload encountered errors")

    return {"pip_upload": pip_success, "success": True}

def step4_publish_environment(workspace_id: str, environment_id: str, environment_name: str, whl_uploads: list, pip_upload: bool) -> Dict[str, Any]:
    """
    Step 4: Publish Environment
    Initiate publish operation to activate staged libraries.

    Args:
        workspace_id: Target workspace ID
        environment_id: Environment ID
        environment_name: Environment name
        whl_uploads: List of WHL upload results
        pip_upload: Pip upload success status

    Returns:
        Dictionary with publish initiation status
    """
    print("\n" + "=" * 60)
    print("STEP 4: Publishing Environment")
    print("=" * 60)

    # Check for any failures
    whl_failures = [u["filename"] for u in whl_uploads if not u["success"]]
    if whl_failures or not pip_upload:
        print("\n⚠ WARNING: Some uploads failed:")
        if whl_failures:
            print(f"  - WHL packages failed: {', '.join(whl_failures)}")
        if not pip_upload:
            print(f"  - Pip packages failed")
        print("\nPublishing will proceed with successfully uploaded packages only.")

    publish_initiated = publish_environment(workspace_id=workspace_id, environment_id=environment_id, environment_name=environment_name)
    if publish_initiated:
        print("\n" + "*" * 60)
        print("⚠️  IMPORTANT: PUBLISH INITIATED (NOT COMPLETED)")
        print("*" * 60)
        print("\n** Publishing can take UP TO 1 HOUR to complete! **")
        print("\n** ACTION REQUIRED: **")
        print("  1. Navigate to the Fabric portal")
        print("  2. Go to your workspace and find the environment")
        print("  3. Check the publishing state/status")
        print("  4. Wait for the status to show 'Published' or 'Active'")
        print("  5. DO NOT use the environment until publishing completes")
        print("\n" + "*" * 60)

    return {"publish_initiated": publish_initiated, "success": True}


def start_environment_deployment(workspace_id: str, environment_name: str, library_artifacts_path: str, discovered_whl_packages: list, environment_description: Optional[str] = None) -> Dict[str, Any]:
    """
    Main orchestration function for deploying libraries to Fabric Environment.
    Calls all step functions in sequence and aggregates results.

    Args:
        workspace_id: Target workspace ID
        environment_name: Name of the environment
        library_artifacts_path: Library artifacts directory containing WHL packages and environment.yml
        discovered_whl_packages: List of WHL package filenames
        environment_description: Optional environment description

    Returns:
        Dictionary containing deployment results and status
    """
    # Initialize results dictionary
    results = {
        "environment_id": None,
        "whl_uploads": [],
        "pip_upload": False,
        "publish_success": False,
        "overall_success": False
    }

    # ========================================
    # Call Step 1: Environment Setup
    # ========================================
    step1_result = step1_setup_environment(
        workspace_id=workspace_id,
        environment_name=environment_name,
        environment_description=environment_description
    )

    results["environment_id"] = step1_result["environment_id"]

    if not step1_result["success"] or not step1_result["environment_id"]:
        return results

    environment_id = step1_result["environment_id"]

    # ========================================
    # Call Step 2: Upload WHL Packages
    # ========================================
    step2_result = step2_upload_whl_packages(
        workspace_id=workspace_id,
        environment_id=environment_id,
        library_artifacts_path=library_artifacts_path,
        discovered_whl_packages=discovered_whl_packages
    )

    results["whl_uploads"] = step2_result["whl_uploads"]

    # ========================================
    # Call Step 3: Upload Pip Packages
    # ========================================
    step3_result = step3_upload_pip_packages(
        workspace_id=workspace_id,
        environment_id=environment_id,
        library_artifacts_path=library_artifacts_path
    )

    results["pip_upload"] = step3_result["pip_upload"]

    # ========================================
    # Call Step 4: Publish Environment
    # ========================================
    step4_result = step4_publish_environment(
        workspace_id=workspace_id,
        environment_id=environment_id,
        environment_name=environment_name,
        whl_uploads=results["whl_uploads"],
        pip_upload=results["pip_upload"]
    )

    results["publish_success"] = step4_result["publish_initiated"]

    # ========================================
    # Final Summary
    # ========================================
    whl_failures = [u["filename"] for u in results["whl_uploads"] if not u["success"]]

    if results["publish_success"] and not whl_failures:
        results["overall_success"] = True
    elif results["publish_success"]:
        results["overall_success"] = False
    else:
        results["overall_success"] = False

    return results

---

## Step 4: Execute Deployment

**Run this cell to execute the complete deployment workflow.**

This cell uses the validated configuration from Step 1 to orchestrate the complete deployment.

**Prerequisites:**
- ✓ Configuration must be successfully completed in Step 1
- ✓ WHL packages must be discovered (or exception would have been raised)
- ✓ Library artifacts path must exist (or exception would have been raised)

**Deployment Process:**
The `deploy_libraries_to_environment()` function coordinates:
- ✓ Set up or verify the Fabric Environment
- ✓ Upload all discovered WHL packages with validation
- ✓ Upload pip dependencies from environment.yml
- ✓ Publish the environment to apply changes
- ✓ Provide a comprehensive deployment summary

**Expected Output:**
- Step-by-step progress for each operation
- Upload validation with file hashes
- Deployment summary with success/failure status
- Next steps for verifying the deployment

**Note:** If you reach this step, configuration validation has passed. All operations are coordinated automatically with proper error handling and status reporting.


In [ ]:
# Execute the complete deployment workflow using configuration from Step 1
deployment_results = start_environment_deployment(
    workspace_id=workspace_id,
    environment_name=environment_name,
    library_artifacts_path=library_artifacts_path,
    discovered_whl_packages=discovered_whl_packages,
    environment_description="Healthcare libraries environment with HDS and DTT packages"
)

# Display final results
print("\n" + "=" * 60)
print("DEPLOYMENT RESULTS")
print("=" * 60)
print(f"Overall Success: {'✓ Yes' if deployment_results['overall_success'] else '✗ No'}")
print(f"Environment ID: {deployment_results['environment_id']}")
print(f"WHL Uploads: {sum(1 for u in deployment_results['whl_uploads'] if u['success'])}/{len(deployment_results['whl_uploads'])} successful")
print(f"Pip Upload: {'✓ Success' if deployment_results['pip_upload'] else '✗ Failed or Skipped'}")
print(f"Publish Initiated: {'✓ Yes (IN PROGRESS)' if deployment_results['publish_success'] else '✗ Failed'}")
print("=" * 60)

if deployment_results['publish_success']:
    print("\n" + "*" * 60)
    print("⚠️  ATTENTION: PUBLISHING IS IN PROGRESS (NOT COMPLETE)")
    print("*" * 60)
    print("\n** This can take UP TO 1 HOUR! **")
    print("\n** GO TO FABRIC PORTAL NOW and: **")
    print("  • Navigate to your workspace")
    print("  • Find the environment in Environments section")
    print("  • Check publishing status")
    print("  • Wait for 'Published' or 'Active' status")
    print("  • DO NOT use the environment until complete")
    print("\n" + "*" * 60)